# Lecture 4 Optional Tutorial: De-identifying clinical text

In the first tutorial we cleaned and explored radiology reports. Real reports contain patient identifiers (names, NHS numbers, dates of birth, addresses).
Before text can be shared for research or model training, those identifiers must be
removed.

We will compare two approaches for text deidentification:

- **Detection-based removal**: use **Microsoft Presidio**, a detection-based tool (NER + regex) that finds
  identifiers it has been taught to recognise.
- **Knowledge-based scrubbing**: remove a patient's known identifiers (pulled from
  structured EHR fields) from the text. This is the approach used by tools like **CRATE**
  (Clinical Records Anonymisation and Text Extraction).

## What we will cover

| Part | Topic | 
|------|-------|
| 1 | Setup, install Presidio | 
| 2 | Why de-identify? (the rules) |
| 3 | Naive regex redaction |
| 4 | Presidio Analyzer | 
| 5 | Presidio Anonymizer | 
| 6 | Custom recognizers (postcode, MRN, GMC) |
| 7 | Knowledge-based scrubbing | 
| 8 | Hybrid pipeline over the whole dataset | 


<font color="red">The cells below red text are part of exercises and may not run properly without inputting further code. </font>

<font color="green">If you prefer to just run the cells and follow what is happening in the exercises, click the toggle below green text to reveal the correct code. </font>

---
## Part 0: Setup

We install **Presidio** (the analyzer + anonymiser) and a small spaCy model.

> We reuse `en_core_web_sm` to keep the download small. For production de-identification
> a larger model (`en_core_web_lg`) gives better name recall.

In [ ]:
import subprocess, sys

# Install Presidio 
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'presidio-analyzer', 'presidio-anonymizer', 'gdown'], check=True)

subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=True)

import pandas as pd
import re
from pathlib import Path

# Presidio building blocks
from presidio_analyzer import AnalyzerEngine, Pattern, PatternRecognizer
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# Tell Presidio to use the small spaCy model 
nlp_config = {'nlp_engine_name': 'spacy',
              'models': [{'lang_code': 'en', 'model_name': 'en_core_web_sm'}]}
nlp_engine = NlpEngineProvider(nlp_configuration=nlp_config).create_engine()

analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=['en'])
anonymizer = AnonymizerEngine()

---
## Part 1: Load the (synthetic) identifiable dataset

The cleaned dataset from Tutorial 1 was already de-identified. To practise
de-identification we need text that contains identifiers, so we use a prepared
variant of those reports with synthetic personal data injected into them (names,
NHS numbers, dates of birth, hospital numbers, addresses, phone numbers, clinician
names)


In [ ]:
import gdown

#load the dataset
Path('./data').mkdir(exist_ok=True)                                  
IDENT_FILE = Path('./data/synthetic_hand_wrist_xray_reports_identifiable.csv')
gdown.download(id='1Sf1QcqQPpO3CK0cxbb0hlBgQ_dy_67Vq', output=str(IDENT_FILE), quiet=False)

df = pd.read_csv(IDENT_FILE)                                       
print(f'Loaded {len(df)} reports with columns:')
print(list(df.columns))

Notice the dataset now has two kinds of column:

| Kind | Columns | Used for |
|------|---------|----------|
| **Structured identifiers** | `patient_name`, `nhs_number`, `dob`, `mrn`, `address`, `phone`, `gp_name` | the knowledge-based scrubber (Part 7) |
| **Free text** | `report` | the de-identification target |

Let's print one full report and see how the identifiers are scattered through the text.

In [ ]:
# Print the first report so we can see the identifiers embedded in free text
print(df.loc[0, 'report'])

Looking at the report above, which *direct* and
*indirect* identifiers can you see?

---
## Part 2: Naive regex 

The obvious first attempt: write a regular expression for each identifier format.
This works for structured identifiers (an NHS number always looks like one) but
quickly breaks down for names which have no fixed format.

In [ ]:
# A couple of regex patterns for clearly-structured identifiers
NHS_RE = re.compile(r'\b\d{3}\s?\d{3}\s?\d{4}\b')      # 10 digits, optional spaces
PHONE_RE = re.compile(r'\b07\d{3}\s?\d{6}\b')            # UK mobile

def naive_redact(text):
    text = NHS_RE.sub('[NHS]', text)                         # Mask anything shaped like an NHS number
    text = PHONE_RE.sub('[PHONE]', text)                     # Mask anything shaped like a mobile
    return text

print(naive_redact(df.loc[0, 'report']))

The NHS number and phone are gone but the name, DOB, address, postcode, MRN and
clinician names are all still there. 

---
## Part 4: Presidio Analyzer: detecting PII

Presidio's Analyzer runs many recognisers over the text: regex recognisers for
structured identifiers (NHS number, credit card, email …) and a spaCy NER
recognizer that finds `PERSON`, `LOCATION`, `DATE_TIME` etc. 

In [ ]:
text = df.loc[0, 'report']
results = analyzer.analyze(text=text, language='en')         # Returns a list of detected spans

# Show each detection with the actual text it covers and the confidence score
print(f'{"ENTITY":15s} {"SCORE":6s} TEXT')
for r in sorted(results, key=lambda x: x.start):
    print(f'{r.entity_type:15s} {r.score:<6.2f} {text[r.start:r.end]!r}')

Look closely at what Presidio did:

- **Correct:** the patient name, NHS number (built-in UK recognizer), dates and phone.

- **Missed:** the postcode and the hospital number (MRN): the hospital number is
  mislabelled as a `US_DRIVER_LICENSE`. Presidio doesn't know UK-local formats out of the box.

- **Over-scrubbing:** clinical terms get caught as identifiers. For instance here `CMC`
  (the carpometacarpal joint) was flagged as an `ORGANIZATION`.


### Exercise 4.1. Run it on a harder report

Report index **4** mentions another clinician by name in the body text. Run the analyzer
on it and check whether predisio found it correctly. 


<font color="red">If you wish to code, this cell contains a helpful structure:</font>

In [ ]:
text = df.loc[4, 'report']            # Pick a different report
results = analyzer.analyze(text=text, language='en')

# TODO: print each detection like we did above and compare against the raw text
print(text)

<font color="green">Example code to obtain the answer is below:</font>

In [ ]:
#@title Click to reveal code
text = df.loc[4, 'report']
results = analyzer.analyze(text=text, language='en')
for r in sorted(results, key=lambda x: x.start):            
    print(f'{r.entity_type:15s} {r.score:<6.2f} {text[r.start:r.end]!r}')
# The patient and clinician names are both caught (NER generalises to unseen names),
#  the postcode and MRN are still missed.

---
## Part 5: Presidio Anonymizer 

Detection only tells us where the identifiers are. The Anonymizer decides what to
do with each one. Presidio contains several operators:

| Operator | Effect | Example |
|----------|--------|---------|
| `redact` | delete the span | `John Smith` → `` |
| `replace` | swap for a placeholder | `John Smith` → `<PERSON>` |
| `mask` | hide some characters | `433 218 1964` → `*** *** 1964` |
| `hash` | one-way hash | `John Smith` → `a1b2c3…` |

You can set a different operator per entity type.

In [ ]:
text = df.loc[0, 'report']
results = analyzer.analyze(text=text, language='en')

# Choose how to treat each entity type
operators = {
    'DEFAULT':      OperatorConfig('replace', {'new_value': '[REDACTED]'}),   
    'PERSON':       OperatorConfig('replace', {'new_value': '<NAME>'}),       # placeholder
    'UK_NHS':       OperatorConfig('mask', {'masking_char': '*', 'chars_to_mask': 8,
                                            'from_end': False}),             # keep last digits
    'DATE_TIME':    OperatorConfig('replace', {'new_value': '<DATE>'}),
}

out = anonymizer.anonymize(text=text, analyzer_results=results, operators=operators)
print(out.text)

### Exercise 5.1. Hash the names instead

Change the `PERSON` operator so names are **hashed** (`hash`) rather than replaced with a
fixed placeholder. Why might a hospital prefer a hash over `<NAME>` when linking records?

<font color="red">If you wish to code, edit the operators dictionary above. 

<font color="green">Example code to obtain the answer is below:</font>

In [ ]:
#@title Click to reveal code
operators = {
    'DEFAULT': OperatorConfig('replace', {'new_value': '[REDACTED]'}),
    'PERSON':  OperatorConfig('hash', {'hash_type': 'sha256'}),   
}
out = anonymizer.anonymize(text=df.loc[0, 'report'],
                           analyzer_results=analyzer.analyze(text=df.loc[0, 'report'], language='en'),
                           operators=operators)
print(out.text[:300])
# A hash is *consistent*: the same name always maps to the same value so you can still
# tell that two reports mention the same person without revealing who they are.

---
## Part 6: Custom recognizers (postcode, MRN, GMC)

Part 4 showed Presidio missing UK-local identifiers. We teach it new ones by adding
`PatternRecognizer`s.

In [ ]:
# A UK postcode pattern and a local hospital-number pattern
postcode_rec = PatternRecognizer(
    supported_entity='UK_POSTCODE',
    patterns=[Pattern('postcode', r'\b[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}\b', 0.9)])

mrn_rec = PatternRecognizer(
    supported_entity='HOSPITAL_NO',
    patterns=[Pattern('mrn', r'\bRJ1\d{6}\b', 0.9)])

analyzer.registry.add_recognizer(postcode_rec)              # Register the new recognizers
analyzer.registry.add_recognizer(mrn_rec)

text = df.loc[0, 'report']
results = analyzer.analyze(text=text, language='en')
new_hits = [(r.entity_type, text[r.start:r.end]) for r in results
            if r.entity_type in ('UK_POSTCODE', 'HOSPITAL_NO')]
print('Now detected:', new_hits)

---
## Part 7: Knowledge-based (denylist) scrubbing

Presidio works without knowing the patient which great for generalising but it can miss
or mistype identifiers as seen in the previous sections. 


Knowledge-based scrubbing takes the opposite approach. Because the EHR already
stores each patient's name, NHS number, DOB, address in structured fields,
we build a per-patient scrubber from those known values and remove every
occurrence from the free text. This is the approach behind tools like **CRATE** (Clinical Records Anonymisation and Text Extraction).

Let's build a simplified scrubber from our structured columns.

In [ ]:
def scrub_known_identifiers(text, row, placeholder='[__]'):
    """Remove a patient's KNOWN identifiers (from structured fields) from free text."""
    fields = [row['patient_name'], row['nhs_number'], row['dob'], row['mrn'],
              row['address'], row['phone'], row['gp_name']]
    tokens = set()
    for f in fields:
        tokens.add(str(f))                                  # the whole field value
        for part in re.split(r'[,\s]+', str(f)):           # and each word within it
            if len(part) >= 3 and part.lower() not in {'the', 'ms', 'mr', 'dr'}:
                tokens.add(part)
    for t in sorted(tokens, key=len, reverse=True):
        text = re.sub(r'(?<!\w)' + re.escape(t) + r'(?!\w)', placeholder, text)
    return text

print(scrub_known_identifiers(df.loc[4, 'report'], df.loc[4]))

The scrubber properly removed the patient's own identifiers, including the postcode and MRN
that the vanilla Presidio pipeline missed.

The reporting radiologist's name is still there: it isn't in the patient's structured
fields which explains why a pure knowledge-based scrubber can't see it.

| | **Presidio** (detection) | **Knowledge-based scrub** |
|---|---|---|
| Needs to know the patient first? | No | Yes (structured fields) |
| Catches unknown identifiers? | Often | No |
| Catches the patient's own identifiers reliably? | Sometimes misses formats | Yes by construction |
| Risk | over-scrub clinical terms | misses anything not in structured data |



---
## Part 8: Hybrid pipeline over the whole dataset

Best practice: run the knowledge-based scrubber first (guarantees the patient's own
identifiers go) then Presidio to catch everything else (clinicians, stray dates).
Let's apply it to all reports.

In [ ]:
def deidentify(row):
    scrubbed = scrub_known_identifiers(row['report'], row)                       # 1) remove known identifiers
    results = analyzer.analyze(text=scrubbed, language='en')         # 2) detect what's left
    out = anonymizer.anonymize(                                      # 3) replace remaining PII
        text=scrubbed, analyzer_results=results,
        operators={'DEFAULT': OperatorConfig('replace', {'new_value': '[REDACTED]'})})
    return out.text

df['report_deid'] = df.apply(deidentify, axis=1)                     # Apply to every report

# Save only the deidentified columns
safe = df[['fake_patient_id', 'scan_datetime', 'report_datetime', 'report_deid']]
safe.to_csv('./data/reports_deidentified.csv', index=False)
print('Saved de-identified dataset. Example:\n')
print(df.loc[4, 'report_deid'])

### What we built

```
identifiable reports (names, NHS numbers, addresses in free text)
  - knowledge-based scrub -> removes the patient's KNOWN identifiers
  - Presidio analyze    -> detects remaining PII 
  - Presidio anonymize  -> replaces it with placeholders
  - drop structured ID columns

```
